# TactiVision: ทดสอบโมเดลเดิมเทียบกับโมเดลหลัง Fine-tune
เลือก Runtime → Change runtime type → T4 GPU แล้วรันเซลล์จากบนลงล่าง

อัปโหลด **football-v2-test-ready.zip** ไฟล์เดียว มี Dataset และ best.pt จากรอบที่เทรนเสร็จแล้ว

Notebook นี้ประเมิน **Test 30 ภาพจากแมตช์ idq95Xp5RI0 เท่านั้น** ใช้ split='test' และไม่มีคำสั่งเทรน

ไฟล์ Train/Validation ใน ZIP เป็นชุดเดิมที่ตรึงไว้เพื่อให้โครงสร้าง Dataset ครบ ไม่ถูกใช้คำนวณคะแนนรอบนี้

ทดสอบทั้งสองโมเดลด้วย imgsz=1280, batch=1, conf=0.001, iou=0.7 และวาดภาพเทียบที่ conf=0.15 เท่ากัน ตามค่าที่ใช้ใน Validation ก่อนหน้า

อย่าเลือกค่าตั้งใหม่จากผล Test แล้วรายงานคะแนนชุดเดิมว่าเป็นการทดสอบอิสระ หากต้องพัฒนาเพิ่มให้เก็บผลรอบนี้ไว้และใช้ Test ใหม่สำหรับการสรุปครั้งถัดไป

คู่มือ API: https://docs.ultralytics.com/modes/val/

In [ ]:
%pip install -q 'ultralytics==8.4.177' gdown pyyaml pillow pandas


In [ ]:
from pathlib import Path
from datetime import datetime
from importlib.metadata import version
import gc, hashlib, json, shutil, zipfile
import torch, yaml, gdown, pandas as pd
from ultralytics import YOLO
from google.colab import files
from PIL import Image as PILImage, ImageDraw
from IPython.display import display

assert torch.cuda.is_available(), 'เลือก Runtime → Change runtime type → T4 GPU ก่อน'
print('GPU:', torch.cuda.get_device_name(0))
RUN_ROOT = Path('/content/tactivision_test') / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_ROOT.mkdir(parents=True, exist_ok=False)
EXPECTED = ['ball', 'goalkeeper', 'player', 'referee']
IMGSZ = 1280
DISPLAY_CONF = 0.15


## อัปโหลด football-v2-test-ready.zip
ใช้ไฟล์ที่จัดไว้แล้ว ไม่ใช่ ZIP ดิบจาก Roboflow และไม่ใช่ ZIP ผลเทรนเดิม

In [ ]:
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
assert len(archives) == 1, 'เลือก football-v2-test-ready.zip ไฟล์เดียว'
DATA_ROOT = RUN_ROOT / 'dataset'
DATA_ROOT.mkdir()
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        assert (DATA_ROOT / member.filename).resolve().is_relative_to(DATA_ROOT.resolve()), 'พาธใน ZIP ไม่ถูกต้อง'
    archive.extractall(DATA_ROOT)
del uploaded
DATA_YAML = DATA_ROOT / 'data.yaml'
config = yaml.safe_load(DATA_YAML.read_text())
audit = json.loads((DATA_ROOT / 'test_audit.json').read_text())
manifest = json.loads((DATA_ROOT / 'test_manifest.json').read_text())
assert audit['purpose'] == 'held_out_test_evaluation' and audit['test_images'] == 30
assert config['names'] == EXPECTED and config['test'] == 'test/images'
test_paths = sorted((DATA_ROOT / 'test/images').glob('*.jpg'))
assert len(test_paths) == len(manifest) == 30
for row in manifest:
    assert row['clip'] == 'idq95Xp5RI0'
    image = DATA_ROOT / 'test/images' / row['filename']
    label = DATA_ROOT / 'test/labels' / row['label']
    assert hashlib.sha256(image.read_bytes()).hexdigest() == row['sha256']
    assert hashlib.sha256(label.read_bytes()).hexdigest() == row['label_sha256']
BEST_MODEL = DATA_ROOT / 'models/best.pt'
assert hashlib.sha256(BEST_MODEL.read_bytes()).hexdigest() == audit['best_model_sha256']
config['path'] = str(DATA_ROOT)
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
EVAL_ARGS = dict(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=1, device=0,
                 conf=0.001, iou=0.7, rect=True, augment=False, workers=2, plots=True,
                 seed=42, deterministic=True, project=str(RUN_ROOT))
print('Test พร้อม:', len(test_paths), 'ภาพ พร้อม Label และ best.pt ที่ตรวจ checksum แล้ว')


## ทดสอบโมเดลเดิม
ดาวน์โหลด football-player-detection.pt จากลิงก์ใน Roboflow Sports setup.sh แล้วคำนวณคะแนนบน Test

In [ ]:
BASE_MODEL = RUN_ROOT / 'football-player-detection.pt'
downloaded = gdown.download(id='17PXFNlx-jI7VjVo_vQnB1sONjRyvoB-q', output=str(BASE_MODEL), quiet=False)
assert downloaded and BASE_MODEL.is_file(), 'ดาวน์โหลดโมเดลเดิมไม่สำเร็จ'
def metric_summary(metrics):
    per_class = {}
    for position, class_id in enumerate(metrics.box.ap_class_index):
        per_class[EXPECTED[int(class_id)]] = {
            'precision': float(metrics.box.p[position]), 'recall': float(metrics.box.r[position]),
            'mAP50': float(metrics.box.ap50[position]), 'mAP50_95': float(metrics.box.ap[position])}
    return {'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
            'mAP50': float(metrics.box.map50), 'mAP50_95': float(metrics.box.map), 'per_class': per_class}
baseline = YOLO(str(BASE_MODEL))
assert [baseline.names[i] for i in range(len(baseline.names))] == EXPECTED
before_metrics = baseline.val(name='baseline_test', **EVAL_ARGS)
before = metric_summary(before_metrics)
(RUN_ROOT / 'baseline_test_metrics.json').write_text(json.dumps(before, indent=2))
display(pd.DataFrame(before['per_class']).T.round(4))
del baseline, before_metrics
gc.collect()
torch.cuda.empty_cache()


## ทดสอบ best.pt หลัง Fine-tune
ใช้ภาพและค่าตั้งเดียวกับโมเดลเดิมทุกอย่าง

In [ ]:
finetuned = YOLO(str(BEST_MODEL))
assert [finetuned.names[i] for i in range(len(finetuned.names))] == EXPECTED
after_metrics = finetuned.val(name='finetuned_test', **EVAL_ARGS)
after = metric_summary(after_metrics)
(RUN_ROOT / 'finetuned_test_metrics.json').write_text(json.dumps(after, indent=2))
display(pd.DataFrame(after['per_class']).T.round(4))
del finetuned, after_metrics
gc.collect()
torch.cuda.empty_cache()


In [ ]:
overall = pd.DataFrame({
    'original': {k:v for k,v in before.items() if isinstance(v, float)},
    'finetuned': {k:v for k,v in after.items() if isinstance(v, float)}})
overall['change'] = overall['finetuned'] - overall['original']
display(overall.round(4))
print('รายคลาส: mAP50-95 (ไม่ใช่เปอร์เซ็นต์ตรวจถูกโดยตรง)')
per_class = pd.DataFrame({
    'original': {k:v['mAP50_95'] for k,v in before['per_class'].items()},
    'finetuned': {k:v['mAP50_95'] for k,v in after['per_class'].items()}}).reindex(EXPECTED)
per_class['change'] = per_class['finetuned'] - per_class['original']
display(per_class.round(4))
overall.to_csv(RUN_ROOT / 'test_overall.csv')
per_class.to_csv(RUN_ROOT / 'test_per_class_map.csv')
report = {'split': 'held_out_test', 'test_images': 30, 'test_clip': 'idq95Xp5RI0',
          'settings': {k:v for k,v in EVAL_ARGS.items() if k not in ('data','project')},
          'display_conf': DISPLAY_CONF, 'before': before, 'after': after,
          'original_model_sha256': hashlib.sha256(BASE_MODEL.read_bytes()).hexdigest(),
          'finetuned_model_sha256': audit['best_model_sha256'],
          'environment': {p:version(p) for p in ('ultralytics','torch','torchvision','numpy','pillow')}}
(RUN_ROOT / 'test_comparison.json').write_text(json.dumps(report, indent=2))
print('นี่คือผล Test ของแมตช์ที่กันไว้ ส่วนคะแนนก่อนหน้านี้เป็น Validation คนละชุดกัน')


## ภาพเทียบซ้ายเดิม / ขวาหลัง Fine-tune
บันทึกครบ 30 คู่ แสดงตัวอย่าง 3 คู่ ใช้ conf=0.15 เท่ากัน ค่านี้กำหนดจากขั้น Validation ก่อนหน้า

Confusion matrix ที่บันทึกจากการวัด mAP ใช้ conf=0.001 จึงอาจมีกรอบความมั่นใจต่ำมากกว่าภาพตัวอย่าง และ Precision/Recall สรุปจากการประเมินอาจไม่ได้มาจาก threshold=0.15

In [ ]:
PAIR_DIR = RUN_ROOT / 'side_by_side_test'
PAIR_DIR.mkdir(exist_ok=True)
old_model = YOLO(str(BASE_MODEL)).to('cpu')
new_model = YOLO(str(BEST_MODEL)).to('cpu')
for index, image_path in enumerate(test_paths):
    panels = []
    for detector in (old_model, new_model):
        result = detector.predict(str(image_path), imgsz=IMGSZ, conf=DISPLAY_CONF, iou=0.7,
                                  device=0, save=False, verbose=False)[0]
        panels.append(PILImage.fromarray(result.plot(line_width=1)[:, :, ::-1]))
        detector.to('cpu')
        torch.cuda.empty_cache()
    width, height = panels[0].size
    pair = PILImage.new('RGB', (width*2, height+35), 'white')
    pair.paste(panels[0], (0,35))
    pair.paste(panels[1], (width,35))
    draw = ImageDraw.Draw(pair)
    draw.text((10,10), 'ORIGINAL - Test', fill='black')
    draw.text((width+10,10), 'FINE-TUNED - Test', fill='black')
    pair.save(PAIR_DIR / image_path.name, quality=95)
    if index < 3:
        print(image_path.name.split('_jpg')[0])
        display(pair.resize((1280,round(pair.height*1280/pair.width))))
print('บันทึกภาพ Test เปรียบเทียบ:', len(test_paths), 'คู่')
del old_model, new_model
gc.collect()
torch.cuda.empty_cache()


## ดาวน์โหลดผล Test
เก็บ ZIP นี้แล้วส่งกลับมาให้ตรวจผลได้ ไม่มีการเทรนหรือเปลี่ยน best.pt ใน Notebook นี้

In [ ]:
BUNDLE = RUN_ROOT / 'test_deliverables'
BUNDLE.mkdir(exist_ok=True)
for name in ('test_comparison.json','test_overall.csv','test_per_class_map.csv',
             'baseline_test_metrics.json','finetuned_test_metrics.json'):
    shutil.copy2(RUN_ROOT / name, BUNDLE / name)
for name in ('test_audit.json','test_manifest.json','training_split_manifest.json'):
    shutil.copy2(DATA_ROOT / name, BUNDLE / name)
for name in ('baseline_test','finetuned_test','side_by_side_test'):
    shutil.copytree(RUN_ROOT / name, BUNDLE / name, dirs_exist_ok=True)
result_zip = shutil.make_archive(str(RUN_ROOT / 'tactivision_test_results'),'zip',BUNDLE)
print('ดาวน์โหลดแล้วเก็บไฟล์ไว้:', result_zip)
files.download(result_zip)
